In [1]:
import os
import sys
import csv
import time
import random
from typing import List, Dict

import numpy as np

# Ensure imports work when running from project root
CURRENT_DIR = os.getcwd() 
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

from dfd import run_deepfake_detection  # Reuse the exact pipeline/fusion logic


def list_video_files(directory_path: str) -> List[str]:
    try:
        return [
            os.path.join(directory_path, f)
            for f in os.listdir(directory_path)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]
    except Exception:
        return []


def sample_videos(files: List[str], sample_size: int, seed: int = 42) -> List[str]:
    rng = random.Random(seed)
    if len(files) <= sample_size:
        rng.shuffle(files)
        return files
    return rng.sample(files, sample_size)


def evaluate_on_celebdf(
    celebd_df_root: str = r"C:\Users\KK\Desktop\dfd\Celeb-DF",
    real_subdir: str = "Celeb-real",
    fake_subdir: str = "Celeb-synthesis",
    num_real: int = 100,
    num_fake: int = 100,
    model_path: str = "model_89_acc_40_frames_final_data.pt",
    run_rppg: bool = True,
    show_visualization: bool = False,
    sequence_length: int = 40,
    frame_rate: int = 40,
    batch_size: int = 30,
    signal_size: int = 270,
    seed: int = 42,
) -> Dict[str, float]:
    real_dir = os.path.join(celebd_df_root, real_subdir)
    fake_dir = os.path.join(celebd_df_root, fake_subdir)

    real_files = list_video_files(real_dir)
    fake_files = list_video_files(fake_dir)

    sampled_real = sample_videos(real_files, num_real, seed)
    sampled_fake = sample_videos(fake_files, num_fake, seed + 1)

    results_rows: List[Dict[str, str]] = []
    counters = {
        "real_total": len(sampled_real),
        "fake_total": len(sampled_fake),
        "real_correct": 0,
        "fake_correct": 0,
    }

    start_time = time.time()

    def process_file(file_path: str, label: str) -> None:
        try:
            result = run_deepfake_detection(
                video_path=file_path,
                model_path=model_path,
                run_rppg=run_rppg,
                show_visualization=show_visualization,
                sequence_length=sequence_length,
                frame_rate=frame_rate,
                batch_size=batch_size,
                signal_size=signal_size,
            )
        except Exception as e:
            result = None
            print(f"Error processing {file_path}: {e}")

        pred = result["prediction"] if result and "prediction" in result else "Unknown"
        conf = float(result["confidence"]) if result and "confidence" in result else 0.0
        cnn_pred = result.get("cnn_rnn_prediction", "Unknown") if result else "Unknown"
        rppg_pred = result.get("rppg_prediction", "Unknown") if result else "Unknown"
        cnn_conf = float(result.get("cnn_rnn_confidence", 0.0)) if result else 0.0
        rppg_conf = float(result.get("rppg_confidence", 0.0)) if result else 0.0
        hr = float(result.get("heart_rate", 0.0)) if result else 0.0
        method = result.get("method", "") if result else ""

        is_correct = 1 if (label.upper() == pred.upper()) else 0
        if label.upper() == "REAL":
            counters["real_correct"] += is_correct
        elif label.upper() == "FAKE":
            counters["fake_correct"] += is_correct

        results_rows.append(
            {
                "file": file_path,
                "label": label,
                "final_prediction": pred,
                "final_confidence": f"{conf:.2f}",
                "cnn_pred": cnn_pred,
                "cnn_conf": f"{cnn_conf:.2f}",
                "rppg_pred": rppg_pred,
                "rppg_conf": f"{rppg_conf:.2f}",
                "heart_rate": f"{hr:.2f}",
                "fusion_method": method,
                "correct": str(is_correct),
            }
        )

    print(f"Found {len(real_files)} real and {len(fake_files)} fake videos.")
    print(f"Sampling {len(sampled_real)} real and {len(sampled_fake)} fake videos for evaluation.\n")

    for idx, fpath in enumerate(sampled_real, 1):
        print(f"[REAL {idx}/{len(sampled_real)}] {fpath}")
        process_file(fpath, label="REAL")

    for idx, fpath in enumerate(sampled_fake, 1):
        print(f"[FAKE {idx}/{len(sampled_fake)}] {fpath}")
        process_file(fpath, label="FAKE")

    duration_min = (time.time() - start_time) / 60.0

    # Aggregate metrics
    real_acc = counters["real_correct"] / counters["real_total"] if counters["real_total"] else 0.0
    fake_acc = counters["fake_correct"] / counters["fake_total"] if counters["fake_total"] else 0.0
    overall_acc = (
        (counters["real_correct"] + counters["fake_correct"]) / (counters["real_total"] + counters["fake_total"]) if (counters["real_total"] + counters["fake_total"]) else 0.0
    )

    # Save CSV
    csv_path = os.path.join(CURRENT_DIR, "celebdf_batch_results.csv")
    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "file",
                "label",
                "final_prediction",
                "final_confidence",
                "cnn_pred",
                "cnn_conf",
                "rppg_pred",
                "rppg_conf",
                "heart_rate",
                "fusion_method",
                "correct",
            ],
        )
        writer.writeheader()
        writer.writerows(results_rows)

    print("\nEvaluation complete.")
    print(f"Results saved to: {csv_path}")
    print(
        f"Real accuracy: {real_acc*100:.2f}% ({counters['real_correct']}/{counters['real_total']}), "
        f"Fake accuracy: {fake_acc*100:.2f}% ({counters['fake_correct']}/{counters['fake_total']}), "
        f"Overall: {overall_acc*100:.2f}%, Time: {duration_min:.1f} min"
    )

    return {
        "real_accuracy": real_acc,
        "fake_accuracy": fake_acc,
        "overall_accuracy": overall_acc,
        "minutes": duration_min,
        "csv": csv_path,
    }

print("hello")

hello


In [ ]:
# Run batch evaluation on Celeb-DF (100 REAL, 100 FAKE)
res = evaluate_on_celebdf(
    celebd_df_root=r"C:\Users\KK\Desktop\dfd\Celeb-DF",
    real_subdir="Celeb-real",
    fake_subdir="Celeb-synthesis",
    num_real=10,
    num_fake=10,
    model_path=os.path.join(CURRENT_DIR, "checkpoint.pt"),
    run_rppg=True,
    show_visualization=False,
    sequence_length=20,
    frame_rate=30,
    batch_size=30,
    signal_size=270,
    seed=42,
)
res


Found 158 real and 794 fake videos.
Sampling 10 real and 10 fake videos for evaluation.

[REAL 1/10] C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-real\id11_0008.mp4


Using cache found in C:\Users\KK/.cache\torch\hub\pytorch_vision_v0.10.0
C:\Users\KK\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\KK\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\KK\Desktop\dfd\rPPG\dfd.py:106: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will exec

CNN-RNN model loaded successfully
Processing video: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-real\id11_0008.mp4
Running CNN-RNN deepfake detection...
Deepfake detection result: FAKE with confidence 50.03%
Running rPPG heart rate analysis...
init


C:\Users\KK\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\KK\Desktop\dfd\rPPG\capture_frames.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no lo

Attempting to open video file: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-real\id11_0008.mp4
Video source opened successfully!
Error in rPPG processing: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1367: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvWaitKey'


DEEPFAKE DETECTION ANALYSIS
CNN-RNN MODEL PREDICTION:
  Result: FAKE
  Confidence: 50.03%

rPPG PHYSIOLOGICAL ANALYSIS:
  Result: Unknown
  Confidence: 0.00%

FUSION RESULT:
  FINAL PREDICTION: FAKE
  Overall Confidence: 50.03%
  Fusion Method: CNN-RNN only (no/unreliable rPPG data)
[REAL 2/10] C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-real\id0_0006.mp4


...

CNN-RNN model loaded successfully
Processing video: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id17_id9_0001.mp4
Running CNN-RNN deepfake detection...
Deepfake detection result: FAKE with confidence 100.00%
Running rPPG heart rate analysis...
init
Attempting to open video file: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id17_id9_0001.mp4
Video source opened successfully!
Error in rPPG processing: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1367: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvWaitKey'


DEEPFAKE DETECTION ANALYSIS
CNN-RNN MODEL PREDICTION:
  Result: FAKE
  Confidence: 100.00%

rPPG PHYSIOLOGICAL ANALYSIS:
  Result: Unknown
  Confidence: 0.00%

FUSION RESULT:
  FINAL PREDICTION: FAKE
  Overall Confidence: 100.00%
  Fusion

Using cache found in C:\Users\KK/.cache\torch\hub\pytorch_vision_v0.10.0


CNN-RNN model loaded successfully
Processing video: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id8_id6_0008.mp4
Running CNN-RNN deepfake detection...
Deepfake detection result: FAKE with confidence 99.87%
Running rPPG heart rate analysis...
init
Attempting to open video file: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id8_id6_0008.mp4
Video source opened successfully!
Error in rPPG processing: OpenCV(4.10.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1367: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvWaitKey'


DEEPFAKE DETECTION ANALYSIS
CNN-RNN MODEL PREDICTION:
  Result: FAKE
  Confidence: 99.87%

rPPG PHYSIOLOGICAL ANALYSIS:
  Result: Unknown
  Confidence: 0.00%

FUSION RESULT:
  FINAL PREDICTION: FAKE
  Overall Confidence: 99.87%
  Fusion Meth

Using cache found in C:\Users\KK/.cache\torch\hub\pytorch_vision_v0.10.0


CNN-RNN model loaded successfully
Processing video: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id9_id4_0005.mp4
Running CNN-RNN deepfake detection...
Deepfake detection result: FAKE with confidence 100.00%
Running rPPG heart rate analysis...
init
Attempting to open video file: C:\Users\KK\Desktop\dfd\Celeb-DF\Celeb-synthesis\id9_id4_0005.mp4
Video source opened successfully!


In [12]:
# Build confusion matrix from saved CSV and visualize
%pip install seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = os.path.join(CURRENT_DIR, "celebdf_batch_results.csv")
df = pd.read_csv(csv_path)

# Normalize labels
true_labels = df["label"].str.upper().fillna("UNKNOWN")
preds = df["final_prediction"].str.upper().fillna("UNKNOWN")

# Map to indices: order [FAKE, REAL]
label_to_idx = {"FAKE": 0, "REAL": 1}

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(true_labels, preds):
    if t in label_to_idx and p in label_to_idx:
        cm[label_to_idx[t], label_to_idx[p]] += 1

# Metrics
tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
overall_acc = (tp + tn) / max(1, cm.sum())
real_acc = tp / max(1, cm[1].sum())
fake_acc = tn / max(1, cm[0].sum())

print(f"Overall accuracy: {overall_acc*100:.2f}%")
print(f"REAL accuracy (recall): {real_acc*100:.2f}%")
print(f"FAKE accuracy (recall): {fake_acc*100:.2f}%")
print("Confusion matrix (rows=true [FAKE, REAL], cols=pred [FAKE, REAL]):")
print(cm)

# Plot
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["FAKE","REAL"], yticklabels=["FAKE","REAL"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Celeb-DF Confusion Matrix (Fusion)")
plt.tight_layout()
plt.show()



[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for seaborn from https://files.pythonhosted.org/packages/83/11/00d3c3dfc25ad54e731d91449895a79e4bf2384dc3ac01809010ba88f6d5/seaborn-0.13.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/294.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/294.9 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/294.9 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 41.0/294.9 kB 279.3 kB/s eta 0:00:01
   -------- ------------------------------ 61.4/294.9 kB 363.1 kB/s eta 0:00:01
   ------------ -------------------------- 92.2/294.9 kB 435.7 kB/s eta 0:00:01
   -------------- ----------------------- 112.6/294.9 kB 502.0 kB/s eta 0:00:01
   ------------------ ------------------- 1

In [13]:
# Confusion matrix with pure Matplotlib (no seaborn)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_path = os.path.join(CURRENT_DIR, "celebdf_batch_results.csv")
df = pd.read_csv(csv_path)

true_labels = df["label"].str.upper().fillna("UNKNOWN")
preds = df["final_prediction"].str.upper().fillna("UNKNOWN")

label_order = ["FAKE", "REAL"]
label_to_idx = {lbl: i for i, lbl in enumerate(label_order)}

cm = np.zeros((2, 2), dtype=int)
for t, p in zip(true_labels, preds):
    if t in label_to_idx and p in label_to_idx:
        cm[label_to_idx[t], label_to_idx[p]] += 1

# Metrics
tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
overall_acc = (tp + tn) / max(1, cm.sum())
real_acc = tp / max(1, cm[1].sum())
fake_acc = tn / max(1, cm[0].sum())
print(f"Overall accuracy: {overall_acc*100:.2f}%")
print(f"REAL recall: {real_acc*100:.2f}%  |  FAKE recall: {fake_acc*100:.2f}%")
print("Confusion matrix (rows=true [FAKE, REAL], cols=pred [FAKE, REAL]):")
print(cm)

# Plot with Matplotlib only
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Ticks and labels
ax.set(xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]))
ax.set_xticklabels(label_order)
ax.set_yticklabels(label_order)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Celeb-DF Confusion Matrix (Fusion)")

# Annotate cells
thresh = cm.max() / 2.0 if cm.max() > 0 else 0.5
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            format(cm[i, j], "d"),
            ha="center",
            va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=11,
        )

fig.tight_layout()
plt.show()


Overall accuracy: 89.00%
REAL recall: 78.00%  |  FAKE recall: 100.00%
Confusion matrix (rows=true [FAKE, REAL], cols=pred [FAKE, REAL]):
[[100   0]
 [ 22  78]]


In [3]:
# Seaborn + sklearn confusion matrix using provided function
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.metrics import confusion_matrix

# Provided function
def print_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print('True positive = ', cm[0][0])
    print('False positive = ', cm[0][1])
    print('False negative = ', cm[1][0])
    print('True negative = ', cm[1][1])
    print('\n')
    df_cm = pd.DataFrame(cm, range(2), range(2))
    sn.set(font_scale=1.4)
    sn.heatmap(df_cm, annot=True, annot_kws={"size": 16})
    plt.ylabel('Actual label', size=20)
    plt.xlabel('Predicted label', size=20)
    plt.xticks(np.arange(2), ['Fake', 'Real'], size=16)
    plt.yticks(np.arange(2), ['Fake', 'Real'], size=16)
    plt.ylim([2, 0])
    plt.show()
    calculated_acc = (cm[0][0] + cm[1][1]) / max(1, (cm[0][0] + cm[0][1] + cm[1][0] + cm[1][1]))
    print("Calculated Accuracy", calculated_acc * 100)

# Build y_true/y_pred from saved CSV (Fake=0, Real=1)
csv_path = os.path.join(CURRENT_DIR, "celebdf_batch_results.csv")
df = pd.read_csv(csv_path)

label_map = {"FAKE": 0, "REAL": 1}
true_labels = df["label"].str.upper().map(label_map)
preds = df["final_prediction"].str.upper().map(label_map)

mask = true_labels.notna() & preds.notna()
y_true = true_labels[mask].astype(int).to_numpy()
y_pred = preds[mask].astype(int).to_numpy()

print_confusion_matrix(y_true, y_pred)


True positive =  10
False positive =  0
False negative =  5
True negative =  5


Calculated Accuracy 75.0


In [ ]:
import os
import sys
import csv
import time
import random
from typing import List, Dict

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Ensure imports work when running from project root
CURRENT_DIR = os.getcwd()
if CURRENT_DIR not in sys.path:
    sys.path.insert(0, CURRENT_DIR)

from dfd import run_deepfake_detection # Reuse the exact pipeline/fusion logic


def list_video_files(directory_path: str) -> List[str]:
    try:
        return [
            os.path.join(directory_path, f)
            for f in os.listdir(directory_path)
            if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
        ]
    except Exception:
        return []


def sample_videos(files: List[str], sample_size: int, seed: int = 42) -> List[str]:
    rng = random.Random(seed)
    if len(files) <= sample_size:
        rng.shuffle(files)
        return files
    return rng.sample(files, sample_size)


def evaluate_on_celebdf(
    celebd_df_root: str = r"C:\Users\KK\Desktop\dfd\Celeb-DF",
    real_subdir: str = "Celeb-real",
    fake_subdir: str = "Celeb-synthesis",
    num_real: int = 100,
    num_fake: int = 100,
    model_path: str = "model_89_acc_40_frames_final_data.pt",
    run_rppg: bool = True,
    show_visualization: bool = False,
    sequence_length: int = 40,
    frame_rate: int = 40,
    batch_size: int = 30,
    signal_size: int = 270,
    seed: int = 42,
) -> Dict[str, float]:
    real_dir = os.path.join(celebd_df_root, real_subdir)
    fake_dir = os.path.join(celebd_df_root, fake_subdir)

    real_files = list_video_files(real_dir)
    fake_files = list_video_files(fake_dir)

    sampled_real = sample_videos(real_files, num_real, seed)
    sampled_fake = sample_videos(fake_files, num_fake, seed + 1)

    results_rows: List[Dict[str, str]] = []
    counters = {
        "real_total": len(sampled_real),
        "fake_total": len(sampled_fake),
        "real_correct": 0,
        "fake_correct": 0,
    }
    
    # --- ADDED: Lists to store data for the ROC curve ---
    y_true: List[int] = []
    y_scores: List[float] = []

    start_time = time.time()

    def process_file(file_path: str, label: str) -> None:
        try:
            result = run_deepfake_detection(
                video_path=file_path,
                model_path=model_path,
                run_rppg=run_rppg,
                show_visualization=show_visualization,
                sequence_length=sequence_length,
                frame_rate=frame_rate,
                batch_size=batch_size,
                signal_size=signal_size,
            )
        except Exception as e:
            result = None
            print(f"Error processing {file_path}: {e}")

        pred = result["prediction"] if result and "prediction" in result else "Unknown"
        conf = float(result["confidence"]) if result and "confidence" in result else 0.0
        
        # --- ADDED: Populate lists for ROC curve calculation ---
        is_correct = 1 if (label.upper() == pred.upper()) else 0
        if label.upper() == "REAL":
            counters["real_correct"] += is_correct
            y_true.append(0) # 0 for the 'negative' class
        elif label.upper() == "FAKE":
            counters["fake_correct"] += is_correct
            y_true.append(1) # 1 for the 'positive' class
        
        # Calculate score for the "positive" class (FAKE)
        score = conf if pred.upper() == "FAKE" else 1.0 - conf
        y_scores.append(score)
        # --- END OF ADDED SECTION ---

        results_rows.append(
            {
                "file": file_path, "label": label, "final_prediction": pred,
                "final_confidence": f"{conf:.2f}",
                "cnn_pred": result.get("cnn_rnn_prediction", "Unknown") if result else "Unknown",
                "cnn_conf": f"{float(result.get('cnn_rnn_confidence', 0.0)):.2f}" if result else "0.00",
                "rppg_pred": result.get("rppg_prediction", "Unknown") if result else "Unknown",
                "rppg_conf": f"{float(result.get('rppg_confidence', 0.0)):.2f}" if result else "0.00",
                "heart_rate": f"{float(result.get('heart_rate', 0.0)):.2f}" if result else "0.00",
                "fusion_method": result.get("method", "") if result else "",
                "correct": str(is_correct),
            }
        )

    print(f"Found {len(real_files)} real and {len(fake_files)} fake videos.")
    print(f"Sampling {len(sampled_real)} real and {len(sampled_fake)} fake videos for evaluation.\n")

    for idx, fpath in enumerate(sampled_real, 1):
        print(f"[REAL {idx}/{len(sampled_real)}] {fpath}")
        process_file(fpath, label="REAL")

    for idx, fpath in enumerate(sampled_fake, 1):
        print(f"[FAKE {idx}/{len(sampled_fake)}] {fpath}")
        process_file(fpath, label="FAKE")

    duration_min = (time.time() - start_time) / 60.0

    # Aggregate metrics
    real_acc = counters["real_correct"] / counters["real_total"] if counters["real_total"] else 0.0
    fake_acc = counters["fake_correct"] / counters["fake_total"] if counters["fake_total"] else 0.0
    overall_acc = (
        (counters["real_correct"] + counters["fake_correct"]) / (counters["real_total"] + counters["fake_total"]) if (counters["real_total"] + counters["fake_total"]) else 0.0
    )

    # --- ADDED: Generate and Save ROC Curve Plot ---
    roc_auc = 0.0
    if y_true and y_scores:
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Classifier (AUC = 0.5)")
        plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
        plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
        plt.title("Receiver Operating Characteristic (ROC) Curve")
        plt.legend(loc="lower right"); plt.grid(True)
        
        plt.savefig(os.path.join(CURRENT_DIR, "roc_curve.pdf"), format="pdf", bbox_inches="tight")
        plt.savefig(os.path.join(CURRENT_DIR, "roc_curve_transparent.png"), format="png", transparent=True, bbox_inches="tight")
        plt.savefig(os.path.join(CURRENT_DIR, "roc_curve.png"), format="png", bbox_inches="tight")
        plt.close()
    # --- END OF ADDED SECTION ---

    # Save CSV
    csv_path = os.path.join(CURRENT_DIR, "celebdf_batch_results.csv")
    if results_rows:
        with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=results_rows[0].keys())
            writer.writeheader()
            writer.writerows(results_rows)

    print("\nEvaluation complete.")
    print(f"Results saved to: {csv_path}")
    if roc_auc > 0: print("ROC curve plots saved successfully.") # ADDED

    # --- MODIFIED: Added AUC to the final summary ---
    print(
        f"Real accuracy: {real_acc*100:.2f}% ({counters['real_correct']}/{counters['real_total']}), "
        f"Fake accuracy: {fake_acc*100:.2f}% ({counters['fake_correct']}/{counters['fake_total']}), "
        f"Overall: {overall_acc*100:.2f}%, AUC: {roc_auc:.2f}, Time: {duration_min:.1f} min"
    )

    # --- MODIFIED: Added AUC to the return dictionary ---
    return {
        "real_accuracy": real_acc,
        "fake_accuracy": fake_acc,
        "overall_accuracy": overall_acc,
        "minutes": duration_min,
        "csv": csv_path,
        "auc": roc_auc,
    }


# Run batch evaluation on Celeb-DF
res = evaluate_on_celebdf(
    celebd_df_root=r"C:\Users\KK\Desktop\dfd\Celeb-DF",
    real_subdir="Celeb-real",
    fake_subdir="Celeb-synthesis",
    num_real=100,
    num_fake=100,
    model_path=os.path.join(CURRENT_DIR, "checkpoint.pt"),
    run_rppg=True,
    show_visualization=False,
    sequence_length=20,
    frame_rate=30,
    batch_size=30,
    signal_size=270,
    seed=42,
)

: 